Holiday Package Prediciton
1) Problem statement.
"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base. One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering * Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information. The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being. However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.

2) Data Collection.
The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction The data consists of 20 column and 4888 rows.

In [ ]:
# importing important libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
df = pd.read_csv("Travel.csv")
df.head()


# Data Cleaning

Handling Missing values

Handling Missing values

Handling Duplicates

Check data type

Understand the dataset

In [ ]:
df.isnull().sum()

In [ ]:
# check all categories
df['Gender'].value_counts()

In [ ]:
df['MaritalStatus'].value_counts()

In [ ]:
df['TypeofContact'].value_counts()

In [ ]:
df['Gender'] = df['Gender'].replace('Fe Male','Female')
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')

In [ ]:
df['Gender'].value_counts()

In [ ]:
df['MaritalStatus'].value_counts()

In [ ]:
df.head()

In [ ]:
## Check Misssing Values
##these are the features with nan value
features_With_na =[features for features in df.columns if df[features].isnull().sum()>=1]
for feature in features_With_na:
    print(feature, np.round(df[feature].isnull().mean()*100,5), '%missing values')

Imputing Null values

Impute Median value for Age column

Impute Mode for Type of Contract

Impute Median for Duration of Pitch

Impute Mode for NumberofFollowup as it is Discrete feature

Impute Mode for PreferredPropertyStar

Impute Median for NumberofTrips

Impute Mode for NumberOfChildrenVisiting

Impute Median for MonthlyIncome

In [ ]:
#Age
df.Age.fillna(df.Age.median(), inplace=True)

#TypeofContract
df.TypeofContact.fillna(df.TypeofContact.mode()[0], inplace=True)

#DurationOfPitch
df.DurationOfPitch.fillna(df.DurationOfPitch.median(), inplace=True)

#NumberOfFollowups
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0], inplace=True)

#PreferredPropertyStar
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0], inplace=True)

#NumberOfTrips
df.NumberOfTrips.fillna(df.NumberOfTrips.median(), inplace=True)

#NumberOfChildrenVisiting
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0], inplace=True)

#MonthlyIncome
df.MonthlyIncome.fillna(df.MonthlyIncome.median(), inplace=True)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop('CustomerID', inplace=True, axis=1)

# Feature Engineering
Feature extraction

In [ ]:
df.head()

In [ ]:
# create a new column for feature
df['TotalVisiting'] = df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'], axis=1, inplace=True)

In [ ]:
## get all the numeric features
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features:', len(num_features))

In [ ]:
# CATEGROICAL FEATURES
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Features:', len(cat_features))

In [ ]:
#Discrete features
discrete_features =[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features:', len(discrete_features))

In [ ]:
#Continious features
continous_features = [feature for feature in num_features if feature not in discrete_features]
print('Num of continous:', len(continous_features))

In [ ]:
df.head()

# Train-Test split and Model Training

In [ ]:
from sklearn.model_selection import train_test_split
X= df.drop(['ProdTaken'], axis=1)
y = df['ProdTaken']

In [ ]:
X.head()

In [ ]:
y.value_counts()

In [ ]:
X.head()

In [ ]:
# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
X.info()

In [ ]:
# Create Column Transformer with 3 types of transformers
cat_features = X.select_dtypes(include="object").columns
num_features = X.select_dtypes(exclude="object").columns

numeric_transformer = StandardScaler()
one_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoding", one_transformer, cat_features),
        ("StandarScaler", numeric_transformer, num_features)
    ]
)

In [ ]:
preprocessor

In [ ]:
## applying Trnsformation in training(fit_transform)
X_train= preprocessor.fit_transform(X_train)

In [ ]:
pd.DataFrame(X_train)

In [ ]:
## apply tansformation on test(transform)
X_test=preprocessor.transform(X_test)

In [ ]:
X_test

# Ada Boost classifier training

In [ ]:
pd.DataFrame(X_train)

In [ ]:
y_train

In [ ]:
# create a function to evaluate model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_sqaure = r2_score(true, predicted)
    return mae, mse, rmse, r2_sqaure

In [ ]:
# Builidng Model Training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge":Ridge(),
    "K-Neighbours Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Ada Boost": AdaBoostRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)# train the model

    #Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #Evaluate Train and Test Dataset
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean SquARED Erroe: {:.4f}".format(model_train_rmse))
    print("-Mean Absoulte Error: {:4f}".format(model_train_mae))
    print("-R2 score: {:.4f}".format(model_train_r2))

    print('-----------------------------------------------------------')

    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))

    print('=' * 35)
    print('\n')

In [ ]:
# Initalize few parameters for Hyperparamter tuning
rf_params = {"max_depth":[5,8,15,None,10],
             "max_features": [5,7,"auto", 8],
             "min_samples_split":[2,8,15,20],
             "n_estimators":[100,200,500,1000]}
adaboost_params ={"n_estimators":[50,60,70,80,90], "algorithm":['SAMME','SAMME.R']}

In [ ]:
rf_params

In [ ]:
adaboost_params

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

# Define the models with their parameters
randomcv_models = [("RF", RandomForestClassifier(), rf_params), ("AB", AdaBoostClassifier(), adaboost_params)]

In [ ]:
randomcv_models

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

In [ ]:
#plot ROC AUC Curve
from sklearn.metrics import roc_auc_score,roc_curve
plt.figure()

#Add the models to the lost that you want to view on the ROC plot
auc_models = [
    {
        'label': 'Adaboost Classifier',
        'model': AdaBoostClassifier(n_estimators=80, algorithm='SAMME'),
        'auc': 0.6049
    },

]

#create a loop through all the model
for algo in auc_models:
    model = algo['model']
    model.fit(X_train, y_train)
    # Compute False postive rate, and True positive rate
    fpr, tpr, thresholds = roc_curve(y_test, model.predict_proba(X_test)[:,1])
# Calculate Area under the curve to display on the plot
    plt.plot(fpr, tpr, label='%s ROC (area = %0.2f)' % (algo['label'], algo['auc']))
# Custom settings for the plot 
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('1-Specificity(False Positive Rate)')
plt.ylabel('Sensitivity(True Positive Rate)')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.savefig("auc.png")
plt.show() 
